# 多風格 AI 繪圖生成器
## Multi-Style AI Image Generator using Hugging Face Diffusers

本 Notebook 展示如何使用 Hugging Face Diffusers 和 Stable Diffusion 創建支持多種藝術風格的 AI 繪圖生成器。
- 支持中文提示詞自動翻譯
- 支持多種預訓練模型風格選擇
- 支持批次生成與網格顯示

## 1. 套件準備 (Import Required Libraries)

In [ ]:
# 【套件準備】導入必要的套件

import torch
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageDraw
from deep_translator import GoogleTranslator
import numpy as np
from typing import List, Tuple

# 顯示 PyTorch 和 GPU 信息
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 風格設定 (Style Models Dictionary)

In [ ]:
# 【風格設定】定義風格對應的模型字典
# Key: 風格名稱 (中文)
# Value: 對應的 Hugging Face Model ID

style_models = {
    '寫實風格': 'runwayml/stable-diffusion-v1-5',
    '二次元動漫': 'hakurei/waifu-diffusion',
    '夢幻藝術': 'dreamlike-art/dreamlike-diffusion-1.0',
    '3D渲染': 'stablediffusionapi/realistic-vision-v30',
    '油畫風格': 'nitrosocial/Ghibli-Diffusion'
}

print("【可用的風格列表】")
for idx, style_name in enumerate(style_models.keys(), 1):
    print(f"{idx}. {style_name}: {style_models[style_name]}")

## 3. 翻譯函式 (Chinese Prompt Translation)

In [ ]:
# 【中文支援】翻譯函式
def translate_prompt(text: str) -> str:
    """
    將中文提示詞翻譯成英文
    
    參數:
        text (str): 中文提示詞
    
    返回:
        str: 翻譯後的英文提示詞
    """
    try:
        # 使用 GoogleTranslator 進行翻譯（中文 -> 英文）
        translator = GoogleTranslator(source_language='zh-CN', target_language='en')
        translated_text = translator.translate(text)
        print(f"【翻譯結果】")
        print(f"原文 (中文): {text}")
        print(f"譯文 (英文): {translated_text}")
        return translated_text
    except Exception as e:
        print(f"翻譯出錯: {e}")
        print(f"使用原文: {text}")
        return text

# 測試翻譯函式
test_prompt = "一個在圖書館看書的藍髮少女"
translated = translate_prompt(test_prompt)

## 4. 圖片網格顯示函式 (Image Grid Display Function)

In [ ]:
# 【批次生成與展示】圖片網格組合函式
def image_grid(imgs: List[Image.Image], rows: int, cols: int) -> Image.Image:
    """
    將多張圖片組合成網格顯示
    
    參數:
        imgs (List[Image.Image]): PIL Image 物件列表
        rows (int): 網格行數
        cols (int): 網格列數
    
    返回:
        Image.Image: 組合後的網格圖片
    """
    # 獲取第一張圖片的尺寸
    w, h = imgs[0].size
    
    # 計算網格的總尺寸
    grid = Image.new('RGB', size=(cols * w, rows * h))
    
    # 將每張圖片貼到對應的位置
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    
    return grid

print("【圖片網格函式已定義】")
print("可以將多張圖片組合成 2x2 的網格進行展示")

## 5. 主要圖片生成函式 (Main Image Generation Function)

In [ ]:
# 【主要函式】生成圖片的完整函式
def generate_images(prompt_text: str, style_name: str, num_images: int = 4) -> Image.Image:
    """
    根據中文提示詞和風格生成 AI 圖片
    
    參數:
        prompt_text (str): 中文提示詞
        style_name (str): 風格名稱（必須在 style_models 字典中）
        num_images (int): 生成的圖片數量（默認 4 張）
    
    返回:
        Image.Image: 組合後的 2x2 網格圖片
    
    異常:
        ValueError: 若選擇的風格不在支持的風格列表中
    """
    
    # 1. 檢查風格是否有效
    if style_name not in style_models:
        raise ValueError(f"不支持的風格: {style_name}。支持的風格有: {list(style_models.keys())}")
    
    print(f"\n{'='*60}")
    print(f"【開始生成圖片】")
    print(f"{'='*60}")
    
    # 2. 翻譯中文提示詞為英文
    print(f"\n[步驟 1/4] 正在翻譯提示詞...")
    english_prompt = translate_prompt(prompt_text)
    
    # 3. 獲取對應風格的模型 ID
    model_id = style_models[style_name]
    print(f"\n[步驟 2/4] 正在載入模型...")
    print(f"風格: {style_name}")
    print(f"模型 ID: {model_id}")
    
    # 4. 載入 Pipeline，配置 GPU 和 float16 以節省記憶體
    try:
        pipeline = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,  # 使用 float16 以節省記憶體
            safety_checker=None          # 禁用安全檢查以加速
        )
        
        # 將模型移至 GPU (CUDA)
        if torch.cuda.is_available():
            pipeline = pipeline.to("cuda")
            print(f"✓ 模型已載入至 GPU")
        else:
            print(f"⚠ GPU 不可用，使用 CPU（速度較慢）")
    
    except Exception as e:
        print(f"❌ 模型載入失敗: {e}")
        raise
    
    # 5. 生成圖片
    print(f"\n[步驟 3/4] 正在生成 {num_images} 張圖片...")
    print(f"提示詞: {english_prompt}")
    
    try:
        # 使用 pipeline 進行推理
        result = pipeline(
            prompt=english_prompt,
            num_images_per_prompt=num_images,
            num_inference_steps=30,      # 推理步數（越多質量越好但越慢）
            guidance_scale=7.5,           # 指導縮放（控制遵守提示的程度）
            height=512,
            width=512
        )
        
        images = result.images
        print(f"✓ 已生成 {len(images)} 張圖片")
    
    except Exception as e:
        print(f"❌ 圖片生成失敗: {e}")
        raise
    
    # 6. 組合成網格
    print(f"\n[步驟 4/4] 正在組合圖片網格...")
    
    # 根據圖片數量確定網格尺寸
    if num_images == 4:
        grid = image_grid(images, rows=2, cols=2)
    elif num_images == 2:
        grid = image_grid(images, rows=1, cols=2)
    else:
        grid = image_grid(images, rows=1, cols=num_images)
    
    print(f"✓ 網格已組合完成")
    print(f"{'='*60}")
    
    # 清除 GPU 記憶體
    del pipeline
    torch.cuda.empty_cache()
    
    return grid

print("【圖片生成函式已定義】")

## 6. 使用範例 (Example Usage)

### 範例：生成「二次元動漫」風格的圖片
**提示詞**：「一個在圖書館看書的藍髮少女」

In [ ]:
# 【使用範例】執行圖片生成

# 設定參數
chinese_prompt = "一個在圖書館看書的藍髮少女"  # 中文提示詞
style = "二次元動漫"                          # 選擇風格

# 調用生成函式
grid_image = generate_images(
    prompt_text=chinese_prompt,
    style_name=style,
    num_images=4
)

## 7. 顯示結果 (Display Results)

In [ ]:
# 【顯示生成的圖片】

print(f"\n【最終結果】")
print(f"風格: {style}")
print(f"提示詞: {chinese_prompt}")
print(f"圖片尺寸: {grid_image.size}")

# 在 Jupyter 中直接顯示圖片
grid_image

## 8. 進階用法 (Advanced Usage)

### 嘗試其他風格和提示詞

In [ ]:
# 【進階示例 1】生成寫實風格的圖片
# 取消注釋下面的程式碼以執行

# prompt_1 = "一個穿著古裝的美女，站在宮殿裡"
# style_1 = "寫實風格"
# grid_1 = generate_images(prompt_1, style_1, num_images=4)
# grid_1

In [ ]:
# 【進階示例 2】生成夢幻藝術風格的圖片
# 取消注釋下面的程式碼以執行

# prompt_2 = "夢幻的森林中一隻發光的鹿"
# style_2 = "夢幻藝術"
# grid_2 = generate_images(prompt_2, style_2, num_images=4)
# grid_2

## 9. 功能總結 (Summary)

### ✅ 實現的功能

| 功能 | 描述 |
|------|------|
| **套件準備** | 導入 torch, diffusers, PIL, deep_translator |
| **風格設定** | 定義 5 種風格模型選擇（寫實、動漫、夢幻、3D、油畫） |
| **中文支援** | 自動翻譯中文提示詞為英文 |
| **模型載入** | float16 精度 + GPU 加速 |
| **批次生成** | 一次生成 4 張圖片 |
| **網格顯示** | 組合為 2x2 網格顯示 |

### 🔧 核心參數說明

- `torch_dtype=torch.float16`: 減少記憶體使用 50%
- `num_inference_steps=30`: 推理步數（影響生成質量）
- `guidance_scale=7.5`: 提示詞引導強度
- `num_images_per_prompt=4`: 每次生成圖片數
- `height=512, width=512`: 生成圖片解析度

### 📝 使用建議

1. 第一次運行需要下載模型（可能需要 5-10 分鐘）
2. 建議使用 NVIDIA GPU 以獲得最佳速度
3. 可自定義 prompt、風格、生成數量
4. 每次生成後自動釋放 GPU 記憶體